常见组合 一 1 先修剪消息，2 再摘要 3 最后记录日志

常见组合 二 1先检查PII 2 限制调用次数 3 总结历史 4 重试工具

执行顺序测试

In [1]:

from dataclasses import dataclass
from typing import Literal, TypedDict, Annotated, Union, Callable

from langchain.agents.middleware import SummarizationMiddleware, HumanInTheLoopMiddleware, PIIMiddleware, \
    ModelCallLimitMiddleware, ModelFallbackMiddleware, LLMToolSelectorMiddleware, ContextEditingMiddleware, \
    ClearToolUsesEdit, FilesystemFileSearchMiddleware, AgentMiddleware, after_agent, wrap_model_call, ModelRequest, \
    ModelResponse, wrap_tool_call
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy, AutoStrategy,    MultipleStructuredOutputsError, StructuredOutputValidationError
from langchain.chat_models import init_chat_model
from langchain_core import messages
from langchain_core.runnables import RunnableConfig
from dotenv import load_dotenv
from langchain.agents import create_agent, AgentState
import os

from langgraph.prebuilt.tool_node import ToolCallRequest
from pydantic import BaseModel, Field
from rich import print as rprint
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
load_dotenv(override=True)
my_api_key = os.getenv("API_KEY")
my_base_url = os.getenv("BASE_URL")
#1 初始化大模型
myModel = init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=300,
    max_retries=3,
    # reasoning_effort="max"#思考模式    #'none', 'minimal', 'low', 'medium', 'high', 'xhigh', 'max'",
)
class Middleware1(AgentMiddleware):
    def before_model(self,state,runtime):
        print("[中间件1] before model--------")
        return None
    def after_model(self,state,runtime):
        print("[中间件1] after model--------")
        return None
class Middleware2(AgentMiddleware):
    def before_model(self,state,runtime):
        print("[中间件2] before model--------")
        return None
    def after_model(self,state,runtime):
        print("[中间件2] after model--------")
        return None
class Middleware3(AgentMiddleware):
    def before_model(self,state,runtime):
        print("[中间件3] before model--------")
        return None
    def after_model(self,state,runtime):
        print("[中间件3] after model--------")
        return None
myAgent=create_agent(
    model=myModel,
    name="agent01",
    tools=[],
    middleware=[Middleware1(),Middleware2(),Middleware3()],
   )


response=myAgent.invoke(
    {
        # 严格校验，使用message少了1个s就会报错
        "messages":
            [
                HumanMessage("你好哇,这是1次测试")
            ],

    },

)
# last_msg=response["messages"][-1]
# rprint(last_msg)
for msg in response["messages"]:
    msg.pretty_print()






[中间件1] before model--------
[中间件2] before model--------
[中间件3] before model--------
[中间件3] after model--------
[中间件2] after model--------
[中间件1] after model--------
================================ Human Message =================================

你好哇,这是1次测试
================================== Ai Message ==================================
Name: agent01

你好哇！收到啦，这是一次测试 ✅


自定义中间件 通过Langchain实现Hook函数（中文称为 钩子函数），指的是在既定的某个特定实际，被框架、系统或主程序自动调用的扩展函数

核心特点 不是主动在业务流程调用 并且 依附于更大的执行流程 作用是在不改变主流远吗的前提下插入自己的逻辑

类型1：Node-style hooks(节点风格钩子)
顾名思义，它们在流程的特定节点运行。
适合顺序逻辑，如记录日志、验证
包括
before_agent：在Agent开始运行之前执行。
before_model：在模型调用之前执行。
after_model：在模型调用之后执行。
after_agent：在Agent流程全部完成后执行。

2种用法 1 装饰器是函数式挂载， 把1个hook快速挂载到agent的某个节点 2 类写法是对象化中间件 封装为1个可配置的、复用的、扩展的组件

    1基于装饰器的函数式挂载，

In [2]:
from langchain.agents.middleware import before_agent,before_model,after_agent,after_model
from typing import  Any
from langgraph.runtime import Runtime
from langchain.agents import AgentState
@before_agent
def before_agent_middleware(state:AgentState,runtime:Runtime)->dict[str,Any]|None:
    state["messages"][-1].content+="---->before agent<----"
    return None
@before_model
def before_model_middleware(state:AgentState,runtime:Runtime)->dict[str,Any]|None:
    state["messages"][-1].content+="---->before model<----"
    return None
@after_model
def after_model_middleware(state:AgentState,runtime:Runtime)->dict[str,Any]|None:
    state["messages"][-1].content+="---->after model<----"
    return None
@after_agent
def after_agent_middleware(state:AgentState,runtime:Runtime)->dict[str,Any]|None:
    state["messages"][-1].content+="---->after_agent<----"
    return None

In [3]:

from dataclasses import dataclass
from typing import Literal, TypedDict, Annotated, Union

from langchain.agents.middleware import SummarizationMiddleware, HumanInTheLoopMiddleware, PIIMiddleware,   ModelCallLimitMiddleware,ModelFallbackMiddleware,LLMToolSelectorMiddleware,ContextEditingMiddleware,ClearToolUsesEdit,FilesystemFileSearchMiddleware
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy, AutoStrategy,    MultipleStructuredOutputsError, StructuredOutputValidationError
from langchain.chat_models import init_chat_model
from langchain_core.runnables import RunnableConfig
from dotenv import load_dotenv
from langchain.agents import create_agent
import os
from pydantic import BaseModel, Field
from rich import print as rprint
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
load_dotenv(override=True)
my_api_key = os.getenv("API_KEY")
my_base_url = os.getenv("BASE_URL")
#1 初始化大模型
myModel = init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=300,
    max_retries=3,
    # reasoning_effort="max"#思考模式    #'none', 'minimal', 'low', 'medium', 'high', 'xhigh', 'max'",
)

myAgent=create_agent(
    model=myModel,
    name="agent01",
    tools=[],
    middleware=[     before_agent_middleware,before_model_middleware,after_model_middleware,after_agent_middleware    ]
   )


response=myAgent.invoke(
    {
        # 严格校验，使用message少了1个s就会报错
        "messages":
            [
                HumanMessage("这是1个基于装饰器实现的自定义中间件执行顺序测试")
            ],

    },

)
# last_msg=response["messages"][-1]
# rprint(last_msg)
for msg in response["messages"]:
    msg.pretty_print()






================================ Human Message =================================

这是1个基于装饰器实现的自定义中间件执行顺序测试---->before agent<-------->before model<----
================================== Ai Message ==================================
Name: agent01

收到。

按你这条信息看，当前中间件执行顺序表达为：

```text
before agent ----> before model
```

也就是：

```text
自定义中间件执行顺序测试
        ↓
before agent
        ↓
before model
```

如果这是“基于装饰器实现的自定义中间件”测试，那么一般对应的执行链路可以理解为：

```python
@agent_middleware
@model_middleware
def run():
    ...
```

执行顺序通常是：

```text
before agent
before model
目标函数 / 后续逻辑
```

如果你的意图是继续测试，我可以配合输出例如：

```text
----> before agent ----> before model ----> target ----> after model ----> after agent
```---->after model<-------->after_agent<----


2 基于类的自定义中间件的实现

In [4]:

from dataclasses import dataclass
from typing import Literal, TypedDict, Annotated, Union

from langchain.agents.middleware import SummarizationMiddleware, HumanInTheLoopMiddleware, PIIMiddleware,   ModelCallLimitMiddleware,ModelFallbackMiddleware,LLMToolSelectorMiddleware,ContextEditingMiddleware,ClearToolUsesEdit,FilesystemFileSearchMiddleware
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy, AutoStrategy,    MultipleStructuredOutputsError, StructuredOutputValidationError
from langchain.chat_models import init_chat_model
from langchain_core.runnables import RunnableConfig
from dotenv import load_dotenv
from langchain.agents import create_agent
import os
from pydantic import BaseModel, Field
from rich import print as rprint
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import before_agent,before_model,after_agent,after_model
from typing import  Any
from langgraph.runtime import Runtime
from langchain.agents import AgentState
from langchain.agents.middleware import AgentMiddleware
load_dotenv(override=True)
my_api_key = os.getenv("API_KEY")
my_base_url = os.getenv("BASE_URL")
#1 初始化大模型
myModel = init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=300,
    max_retries=3,
    # reasoning_effort="max"#思考模式    #'none', 'minimal', 'low', 'medium', 'high', 'xhigh', 'max'",
)
#基于类的中间件自定义实现方式
class MyMiddleware(AgentMiddleware):
    def before_agent(self,state:AgentState,runtime:Runtime)->dict[str,Any]|None:
        state["messages"][-1].content+="---->before agent<----"
        return None

    def before_model(self,state:AgentState,runtime:Runtime)->dict[str,Any]|None:
        state["messages"][-1].content+="---->before model<----"
        return None

    def after_model(self,state:AgentState,runtime:Runtime)->dict[str,Any]|None:
        state["messages"][-1].content+="---->after model<----"
        return None

    def after_agent(self,state:AgentState,runtime:Runtime)->dict[str,Any]|None:
        state["messages"][-1].content+="---->after_agent<----"
        return None
my_middleware=MyMiddleware()
myAgent=create_agent(
    model=myModel,
    name="agent01",
    tools=[],
    middleware=[     my_middleware    ]
   )
response=myAgent.invoke(
    {
        # 严格校验，使用message少了1个s就会报错
        "messages":
            [
                HumanMessage("你好")
            ],
    },
)

for msg in response["messages"]:
    msg.pretty_print()






================================ Human Message =================================

你好---->before agent<-------->before model<----
================================== Ai Message ==================================
Name: agent01

你好！我是 Qwen3（通义千问 3），很高兴为您提供帮助。  

如果您想了解关于 **Agent（智能体）** 或 **Model（模型）** 的技术细节，或者有其他问题需要解答，请随时告诉我具体需求，我会尽力协助您！ 😊---->after model<-------->after_agent<----


before_model 通常的场景：
消息修剪（trim messages）
PII 脱敏
输入验证
条件路由
after_model 通常的场景：
输出验证
格式化响应
统计信息
状态更新

装饰器参数 can_jump_to    Node_Style的4个钩子函数都可以接收额外的参数can_jump_to，钩子函数可以改变agent正常运行轨迹，当发现上下文窗口溢出的时候，则直接跳转至结尾。有3个取值 end,tools,model,

类型2：Wrap-style hooks(包装风格钩子)
顾名思义，它们在模型或工具调用前后运行。
适合控制流，如重试、回退、缓存。
包括wrap_model_call (包裹模型调用)
wrap_tool_call (包裹工具调用)

wrap_model_call 基于装饰器的实现

In [8]:
from dataclasses import dataclass
from typing import Literal, TypedDict, Annotated, Union

from langchain.agents.middleware import SummarizationMiddleware, HumanInTheLoopMiddleware, PIIMiddleware,   ModelCallLimitMiddleware,ModelFallbackMiddleware,LLMToolSelectorMiddleware,ContextEditingMiddleware,ClearToolUsesEdit,FilesystemFileSearchMiddleware
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy, AutoStrategy,    MultipleStructuredOutputsError, StructuredOutputValidationError
from langchain.chat_models import init_chat_model
from langchain_core.runnables import RunnableConfig
from dotenv import load_dotenv
from langchain.agents import create_agent
import os
from pydantic import BaseModel, Field
from rich import print as rprint
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import before_agent,before_model,after_agent,after_model
from typing import  Any,Callable
from langgraph.runtime import Runtime
from langchain.agents import AgentState
from langchain.agents.middleware import ModelCallResult,AgentMiddleware,wrap_model_call,ModelResponse,ModelRequest
load_dotenv(override=True)
my_api_key = os.getenv("API_KEY")
my_base_url = os.getenv("BASE_URL")
#1 初始化大模型
myModel = init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=300,
    max_retries=3,
    # reasoning_effort="max"#思考模式    #'none', 'minimal', 'low', 'medium', 'high', 'xhigh', 'max'",
)

@wrap_model_call
def wrap_model_call_middleware(
         #调用模型之前发送给大模型的数据
        request:ModelRequest,
        #处理器 模型的调用情况
        handler:Callable[[ModelRequest],ModelResponse]
)->ModelResponse|None:
    request.messages[-1].content +="---->wrap model call before <----"
    #模型的调用
    wrap_response=handler(request)
    wrap_response.result[0].content+="---->wrap model call after <----"
    return wrap_response
myAgent=create_agent(
    model=myModel,
    name="agent01",
    tools=[],
    middleware=[     wrap_model_call_middleware    ]
   )
response=myAgent.invoke(
    {
        # 严格校验，使用message少了1个s就会报错
        "messages":
            [
                HumanMessage("你好")
            ],
    },
)

for msg in response["messages"]:
    msg.pretty_print()






================================ Human Message =================================

你好---->wrap model call before <----
================================== Ai Message ==================================
Name: agent01

你好！你是想在“调用模型之前”包一层逻辑吗？下面给你一个通用的 Python 示例：

```python
def wrap_model_call(func):
    def wrapper(*args, **kwargs):
        # 在 model call 之前执行
        print("Before model call: 记录日志 / 校验参数 / 鉴权等")

        # 真正调用模型
        result = func(*args, **kwargs)

        # 在 model call 之后执行
        print("After model call")
        return result

    return wrapper


@wrap_model_call
def call_model(prompt: str):
    # 这里假装是模型调用
    return f"model response to: {prompt}"


print(call_model("你好"))
```

如果你想要的是 **在某个标记 `<----` 之前包装 model call**，也可以告诉我你要包装的具体代码、语言或框架，我可以直接帮你改。---->wrap model call after <----


wrap_model_call 基于装饰类的实现

In [19]:

from dataclasses import dataclass
from typing import Literal, TypedDict, Annotated, Union

from langchain.agents.middleware import SummarizationMiddleware, HumanInTheLoopMiddleware, PIIMiddleware,   ModelCallLimitMiddleware,ModelFallbackMiddleware,LLMToolSelectorMiddleware,ContextEditingMiddleware,ClearToolUsesEdit,FilesystemFileSearchMiddleware
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy, AutoStrategy,    MultipleStructuredOutputsError, StructuredOutputValidationError
from langchain.chat_models import init_chat_model
from langchain_core.runnables import RunnableConfig
from dotenv import load_dotenv
from langchain.agents import create_agent
import os
from pydantic import BaseModel, Field
from rich import print as rprint
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import before_agent,before_model,after_agent,after_model
from typing import  Any,Callable
from langgraph.runtime import Runtime
from langchain.agents import AgentState
from langchain.agents.middleware import ModelCallResult,AgentMiddleware,wrap_model_call,ModelResponse,ModelRequest
load_dotenv(override=True)
my_api_key = os.getenv("API_KEY")
my_base_url = os.getenv("BASE_URL")
#1 初始化大模型
myModel = init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=300,
    max_retries=3,
    # reasoning_effort="max"#思考模式    #'none', 'minimal', 'low', 'medium', 'high', 'xhigh', 'max'",
)
#基于类的中间件自定义实现方式
class WrapModelCallMiddleware(AgentMiddleware):
     # @staticmethod
     def wrap_model_call_middleware(
             #调用模型之前发送给大模型的数据
             self,
            request:ModelRequest,
            #处理器 模型的调用情况
            handler:Callable[[ModelRequest],ModelResponse]
     )->ModelResponse|None:
            request.messages[-1].content +="---->wrap model call before <----"
            print("---->wrap model call before <----")
        #模型的调用
            wrap_response=handler(request)
            wrap_response.result[0].content+="---->wrap model call after <----"
            print("---->wrap model call after <----")
            return wrap_response

myAgent=create_agent(
    model=myModel,
    name="agent01",
    tools=[],
    middleware=[     WrapModelCallMiddleware()    ]
   )
response=myAgent.invoke(
    {
        # 严格校验，使用message少了1个s就会报错
        "messages":

            [ SystemMessage("你是1个专业的Ai助手"),               HumanMessage("你好,请用1句话介绍你自己")            ],
    },

)
for msg in response["messages"]:

    msg.pretty_print()


{
    'messages': [
        SystemMessage(
            content='你是1个专业的Ai助手',
            additional_kwargs={},
            response_metadata={},
            id='2b8657b8-2b8e-41f5-a24f-317ffdb4bbc5'
        ),
        HumanMessage(
            content='你好,请用1句话介绍你自己',
            additional_kwargs={},
            response_metadata={},
            id='fa540fc7-1317-466f-872b-d0daad4b4a46'
        ),
        AIMessage(
            content='我是通义千问，由阿里巴巴集团自主研发的大语言模型，是一个乐于助人的 AI 
伙伴，可以协助你解答问题、提供创意和完成各种任务。😊',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 103,
                    'prompt_tokens': 66,
                    'total_tokens': 169,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 64,
                        'rejected_prediction_tokens': None,
                        'text_tokens': 103
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 0,
                        'image_tokens': None,
                        'text_tokens': 66
                    }
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.8-max',
                'system_fingerprint': None,
                'id': 'chatcmpl-e3ea12b4-9f65-969b-84e4-bb7a3ac13595',
                'finish_reason': 'stop',
                'logprobs': None
            },
            name='agent01',
            id='lc_run--01a0b4fb-9760-7aa0-8d30-bdf848c3e74b-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 66,
                'output_tokens': 103,
                'total_tokens': 169,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 64}
            }
        )
    ]
}

使用场景 用于拦截，重试、响应缓存，修改系统提示

Wrap_Tool_Call基于装饰器的实现

In [18]:

from langgraph.types import Command
from dataclasses import dataclass
from typing import Literal, TypedDict, Annotated, Union

from langchain.agents.middleware import SummarizationMiddleware, HumanInTheLoopMiddleware, PIIMiddleware,   ModelCallLimitMiddleware,ModelFallbackMiddleware,LLMToolSelectorMiddleware,ContextEditingMiddleware,ClearToolUsesEdit,FilesystemFileSearchMiddleware
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy, AutoStrategy,    MultipleStructuredOutputsError, StructuredOutputValidationError
from langchain.chat_models import init_chat_model
from langchain_core.runnables import RunnableConfig
from dotenv import load_dotenv
from langchain.agents import create_agent
import os
from pydantic import BaseModel, Field
from rich import print as rprint
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage,ToolMessage

from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import before_agent,before_model,after_agent,after_model
from typing import  Any,Callable
from langgraph.runtime import Runtime
from langchain.agents import AgentState
from langchain.agents.middleware import ModelCallResult,AgentMiddleware,wrap_model_call,ModelResponse,ModelRequest,wrap_tool_call,ToolCallRequest
load_dotenv(override=True)
my_api_key = os.getenv("API_KEY")
my_base_url = os.getenv("BASE_URL")
#1 初始化大模型
myModel = init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=300,
    max_retries=3,
    # reasoning_effort="max"#思考模式    #'none', 'minimal', 'low', 'medium', 'high', 'xhigh', 'max'",
)
class WeatherInput(BaseModel):
    #如果在函数声明处指定了参数以及参数类型，并且启用了args_schema，那么则以args_schema指定的类型为准
    city: str=Field(
        description="具体的城市名称，例如'北京市'、'重庆市'、''天津市",
        default="北京市",
        min_length=2,
        max_length=10,
    )
    unit:Literal["celsius","fahrenheit","kelvin"]=Field(
        description="描述气温的单位，必须是摄氏度，凯文等单位"
    )
    if_forcast:bool=Field(
        default=True,
        description="是否预测未来1天的天气"
    )
#3 声明1个函数（工具）
@tool(description="获取具体城市的天气",args_schema=WeatherInput)
def get_weather(city:str,unit:str,if_forcast:bool):
    result= f"2026-09-18{city}天气晴朗,9-16{unit},温度适宜，适合出门晒太阳"
    if if_forcast:
        result+="但是明天可能有雨，记得带伞"
    return result

@wrap_tool_call
def wrap_tool_call_middleware(        request:ToolCallRequest,        handler:Callable[   [ToolCallRequest],ToolMessage|Command[Any] ])->ToolMessage | Command[Any] :
    result=handler(request)
    print(f"原始参数：{request.tool_call['args']}")
    request.tool_call["args"]["if_forcast"]=False
    result=handler(request)
    print(f"更新之后参数：{request.tool_call['args']}")
    return result
myAgent=create_agent(
    model=myModel,
    name="agent01",
    tools=[get_weather],
    middleware=[     wrap_tool_call_middleware   ]
   )
response=myAgent.invoke(
    {
        # 严格校验，使用message少了1个s就会报错
        "messages":

            [ SystemMessage("你是1个专业的Ai助手"),               HumanMessage("帮我查询重庆市2026-9-18的天气")            ],
    },

)

for msg in response["messages"]:
    msg.pretty_print()

原始参数：{'city': '重庆市', 'unit': 'celsius', 'if_forcast': True}
更新之后参数：{'city': '重庆市', 'unit': 'celsius', 'if_forcast': False}
================================ System Message ================================

你是1个专业的Ai助手
================================ Human Message =================================

帮我查询重庆市2026-9-18的天气
================================== Ai Message ==================================
Name: agent01

我来帮您查询重庆市的天气信息。
Tool Calls:
  get_weather (call_81bbc23e24514f59bd828bbf)
 Call ID: call_81bbc23e24514f59bd828bbf
  Args:
    city: 重庆市
    unit: celsius
    if_forcast: False
================================= Tool Message =================================
Name: get_weather

2026-09-18重庆市天气晴朗,9-16celsius,温度适宜，适合出门晒太阳
================================== Ai Message ==================================
Name: agent01

以下是重庆市 **2026年9月18日** 的天气情况：

## 🌤️ 重庆市天气预报

| 项目 | 详情 |
|------|------|
| **日期** | 2026年9月18日 |
| **天气状况** | 晴朗 ☀️ |
| **温度范围** | 9℃ ~ 16℃ |
| **温馨提示** | 温度适宜，适合出门晒太阳 |


Wrap_Tool_Call基于类的实现

In [20]:

from langgraph.types import Command
from dataclasses import dataclass
from typing import Literal, TypedDict, Annotated, Union

from langchain.agents.middleware import SummarizationMiddleware, HumanInTheLoopMiddleware, PIIMiddleware,   ModelCallLimitMiddleware,ModelFallbackMiddleware,LLMToolSelectorMiddleware,ContextEditingMiddleware,ClearToolUsesEdit,FilesystemFileSearchMiddleware
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy, AutoStrategy,    MultipleStructuredOutputsError, StructuredOutputValidationError
from langchain.chat_models import init_chat_model
from langchain_core.runnables import RunnableConfig
from dotenv import load_dotenv
from langchain.agents import create_agent
import os
from pydantic import BaseModel, Field
from rich import print as rprint
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage,ToolMessage

from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import before_agent,before_model,after_agent,after_model
from typing import  Any,Callable
from langgraph.runtime import Runtime
from langchain.agents import AgentState
from langchain.agents.middleware import ModelCallResult,AgentMiddleware,wrap_model_call,ModelResponse,ModelRequest,wrap_tool_call,ToolCallRequest
load_dotenv(override=True)
my_api_key = os.getenv("API_KEY")
my_base_url = os.getenv("BASE_URL")
#1 初始化大模型
myModel = init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=300,
    max_retries=3,
    # reasoning_effort="max"#思考模式    #'none', 'minimal', 'low', 'medium', 'high', 'xhigh', 'max'",
)
class WeatherInput(BaseModel):
    #如果在函数声明处指定了参数以及参数类型，并且启用了args_schema，那么则以args_schema指定的类型为准
    city: str=Field(
        description="具体的城市名称，例如'北京市'、'重庆市'、''天津市",
        default="北京市",
        min_length=2,
        max_length=10,
    )
    unit:Literal["celsius","fahrenheit","kelvin"]=Field(
        description="描述气温的单位，必须是摄氏度，凯文等单位"
    )
    if_forcast:bool=Field(
        default=True,
        description="是否预测未来1天的天气"
    )
#3 声明1个函数（工具）
@tool(description="获取具体城市的天气",args_schema=WeatherInput)
def get_weather(city:str,unit:str,if_forcast:bool):
    result= f"2026-09-18{city}天气晴朗,9-16{unit},温度适宜，适合出门晒太阳"
    if if_forcast:
        result+="但是明天可能有雨，记得带伞"
    return result

class WrapToolCallMiddleware(AgentMiddleware):
    def wrap_tool_call(  self,      request:ToolCallRequest,        handler:Callable[   [ToolCallRequest],ToolMessage|Command[Any] ])->ToolMessage | Command[Any] :
        result=handler(request)
        print(f"原始参数：{request.tool_call['args']}")
        request.tool_call["args"]["if_forcast"]=False
        result=handler(request)
        print(f"更新之后参数：{request.tool_call['args']}")
        return result
myAgent=create_agent(
    model=myModel,
    name="agent01",
    tools=[get_weather],
    middleware=[     WrapToolCallMiddleware()   ]
   )
response=myAgent.invoke(
    {
        # 严格校验，使用message少了1个s就会报错
        "messages":

            [ SystemMessage("你是1个专业的Ai助手"),               HumanMessage("帮我查询重庆市2026-9-18的天气")            ],
    },

)

for msg in response["messages"]:
    msg.pretty_print()

原始参数：{'city': '重庆市', 'unit': 'celsius', 'if_forcast': True}
更新之后参数：{'city': '重庆市', 'unit': 'celsius', 'if_forcast': False}
================================ System Message ================================

你是1个专业的Ai助手
================================ Human Message =================================

帮我查询重庆市2026-9-18的天气
================================== Ai Message ==================================
Name: agent01

我先为您查询重庆市的天气数据，稍等。
Tool Calls:
  get_weather (call_225a0852cb734d68ae8fc277)
 Call ID: call_225a0852cb734d68ae8fc277
  Args:
    city: 重庆市
    unit: celsius
    if_forcast: False
================================= Tool Message =================================
Name: get_weather

2026-09-18重庆市天气晴朗,9-16celsius,温度适宜，适合出门晒太阳
================================== Ai Message ==================================
Name: agent01

已为您查询到重庆市 2026 年 9 月 18 日的天气情况：

## 📍 重庆市 · 2026-09-18 天气预报

| 项目 | 详情 |
|---|---|
| ☀️ 天气状况 | 晴朗 |
| 🌡️ 气温范围 | 9℃ ~ 16℃ |
| 📅 日期 | 2026年9月18日（星期五）|

### 💡 温馨提示
- **温度

一般用于监控、重试、修改工具的执行

装饰器和类的选择
当一个中间件只需要实现一个钩子函数时，直接使用装饰器最简单。
当一个中间件需要实现多个钩子函数时，类写法更合适。